# Pythonic recipe builder example

Shows how to author a recipe in Python and serialize it to the same JSON/YAML schema used by stores, catalogs, and the CLI.

In [ ]:
import numpy as np

import woodpecker
from woodpecker.recipes import document, fix, recipe
from woodpecker.testing import make_cmip6

Build a recipe with normal Python calls.

In [ ]:
cmip6_core = recipe(
    "cmip6.core_units",
    fix("woodpecker.normalize_tas_units_to_kelvin"),
    description="Normalize CMIP6 tas units.",
).match(
    dataset_id_patterns=["CMIP6.CMIP.*.Amon.tas.*"],
    attrs={"project_id": "CMIP6", "activity_id": "CMIP"},
)

cmip6_core.to_payload()

Serialize the recipe document. Pass a path to `to_yaml(...)` or `to_json(...)` to write a file.

In [ ]:
print(cmip6_core.to_yaml())

Use `to_model()` to run the recipe immediately.

In [ ]:
dataset = make_cmip6(overrides={"units": "degC"}, seed=7)
original_values = dataset["tas"].values.copy()

plan_model = cmip6_core.to_model()
findings = woodpecker.recipe.check(dataset, plan_model)

findings.fix_ids

In [ ]:
result = woodpecker.recipe.apply(dataset, plan_model, dry_run=True)

(
    result.stats,
    result.preview,
    dataset["tas"].attrs["units"],
    np.allclose(dataset["tas"].values, original_values),
)

In [ ]:
write = woodpecker.recipe.apply(dataset, plan_model, dry_run=False)

(
    write.stats,
    dataset["tas"].attrs["units"],
    np.allclose(dataset["tas"].values, original_values + 273.15),
)

A recipe document can contain multiple Python-authored recipes.

In [ ]:
plan_document = document(
    cmip6_core,
    recipe("c3s.atlas", fix("atlas.encoding_cleanup")).match(path_patterns=["*atlas*.nc"]),
)

plan_document.to_payload()